In [ ]:
import openai, json

client = openai.OpenAI()
messages = []

In [ ]:
def get_weather(city):
    return f"The weather in {city} is sunny"

FUNCTIONS_MAP = {
    "get_weather": get_weather
}

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [ ]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                } for tool_call in message.tool_calls
            ]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = tool_call.function.arguments
            
            print(f"Calling Function: {function_name} with args: {function_args}")
            
            try:
                arguments = json.loads(function_args)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTIONS_MAP.get(function_name)

            result = function_to_run(**arguments)
            # function_args는 '{"city": "Seoul"}' 과 같이 생긴 문자열(모델이 주는건 텍스트이기 때문.)
            # 이걸 Python 딕셔너리로 변환하기 위해서 json.loads() 함수를 사용함.
            # get_weather 함수에 전달할 인자는 딕셔너리가 아니라 도시 이름 하나만 받으면 되기 때문에 ** 을 사용.
            # ** 과 같은 표기법은 해당 딕셔너리를 키워드 인자로 바꿔줌 city="Seoul" 와 같은 형태로.

            print(f"Ran function {function_name} with result: {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })
        
        call_ai()
    else:
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        print(f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
while True:
    message = input("Send a message to the LLM ...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()